In [ ]:
!pip install mediapipe
!pip install ultralytics
!pip install deepface
!pip install git+https://github.com/edavalosanaya/L2CS-Net.git@main

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 

In [ ]:
!wget -O face_landmarker_v2_with_blendshapes.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task
!wget -O face_landmarker_v2_with_blendshapes.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -q -O detector.tflite -q https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys

sys.path.append(
    "/content/drive/MyDrive/Online_Exam_Cheating"
)

In [4]:
import time
from huggingface_hub import hf_hub_download
import cv2

In [5]:
from vision.eye_tracking import EyeTrackingGaze
from vision.objects_detection import ObjectDetection
from vision.head_pose import HeadPoseExtractor
from vision.face_detection import FaceDetector

In [ ]:
!kaggle datasets list
!kaggle datasets download -d raajanwankhade/oep-dataset
!unzip oep-dataset.zip -d /content/oep_dataset

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
Dataset URL: https://www.kaggle.com/datasets/raajanwankhade/oep-dataset
License(s): unknown
100% 11.0G/11.0G [01:28<00:00, 134MB/s] 

Archive:  oep-dataset.zip
  inflating: /content/oep_dataset/OEP database/READ_ME.txt  
  inflating: /content/oep_dataset/OEP database/subject1/Yousef.wav  
  inflating: /content/oep_dataset/OEP database/subject1/Yousef1.avi  
  inflating: /content/oep_dataset/OEP database/subject1/Yousef2.avi  
  inflating: /content/oep_dataset/OEP database/subject1/gt.txt  
  inflating: /content/oep_dataset/OEP database/subject10/gt.txt  
  inflating: /content/oep_dataset/OEP database/subject10/huangpi2.wav  
  inflating: /content/oep_dataset/OEP database/subject10/huangpi21.avi  
  inflating: /content/oep_dataset/OEP database/subject10/huangpi22.avi  
  inflating: /content/oep_dataset/OEP data

In [13]:
import os


os.environ["KAGGLE_API_TOKEN"] = "KGAT_de3dedb37047e7aff1d42e31765cfc6b"


# برای پیدا کردن ویدیو
dataset_path = "/content/oep_dataset/OEP database"

video_paths = []

for subject in sorted(os.listdir(dataset_path)):

    subject_path = os.path.join(dataset_path, subject)

    if os.path.isdir(subject_path):

        videos = [
            f for f in os.listdir(subject_path)
            if f.endswith(".avi")
        ]

        if len(videos) >= 1:
            selected_video = videos[0]

            video_paths.append({
                "subject": subject,
                "video": selected_video,
                "path": os.path.join(subject_path, selected_video)
            })

print(video_paths[2])

{'subject': 'subject11', 'video': 'alhashe31.avi', 'path': '/content/oep_dataset/OEP database/subject11/alhashe31.avi'}


In [15]:

model_asset_path = "face_landmarker_v2_with_blendshapes.task"


detector = ObjectDetection(
    model_path="pose_landmarker.task",
    yolo_model_path="yolov8s.pt",
    earphone_model_path="/content/drive/MyDrive/Online_Exam_Cheating/best.pt"
)


weights = hf_hub_download(
    repo_id="tianfxc/l2cs",
    filename="L2CSNet_gaze360.pkl"
)

tracker = EyeTrackingGaze(
    model_asset_path=model_asset_path,
    weights=weights )

face_detector = FaceDetector(
    model_path="detector.tflite",
)


pose_extractor = HeadPoseExtractor(
    "face_landmarker_v2_with_blendshapes.task"
)

all_sequences = []
objects_boxes = []
dets = []
gaze_result = None
pose = None



for item in video_paths[2:3]:

    video_path = item["path"]
    video_id = item["subject"]

    sequence_data = {
        "video_id": video_id,
        "frames": []
    }

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 30

    frame_id = 0



    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        h, w = frame.shape[:2]

        timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))



        try:

          face_result = face_detector.face_detection(
              int(timestamp_ms),
              frame
          )
        except Exception as e:
          print(
              f"🔴Face detection error at frame {frame_id}:",
              e
          )

        try:
          head_pose = pose_extractor.extract(
              frame,
              timestamp_ms
          )
        except Exception as e:
          print(
              f"🔴Head pose error at frame {frame_id}:",
              e
          )

        if frame_id % 10 == 0:
          try:

              pose = detector.pose_detection(
                  frame,
                  timestamp_ms,
                  w,
                  h
              )


          except Exception as e:

              print(
                  f"🔴Pose error at frame {frame_id}:",
                  e
              )



        if frame_id % 25 == 0:
          try:

              objects_boxes, dets = detector.object_detection(
                  frame
              )

          except Exception as e:

              print(
                  f"🔴Object detection error at frame {frame_id}:",
                  e
              )


        if frame_id % 25 == 0:
          try:

              gaze_result = tracker.extract(
                  frame,
                  timestamp_ms
              )

          except Exception as e:

              print(
                  f"🔴Gaze error at frame {frame_id}:",
                  e
              )



        frame_data = {

            "frame_id": frame_id,

            "timestamp_ms": timestamp_ms,

            "face": face_result,

            "gaze": gaze_result,

            "head_pose": head_pose,

            "objects": [],

            "pose_features": {

                "left_elbow_angle": None,

                "right_elbow_angle": None,

                "left_wrist_distance": None,

                "right_wrist_distance": None,

                "left_elbow_distance": None,

                "right_elbow_distance": None
            }
        }



        for det in dets:

            frame_data["objects"].append({

                "bbox": [
                    float(det[0]),
                    float(det[1]),
                    float(det[2]),
                    float(det[3])
                ],

                "conf": float(det[4]),

                "class": int(det[5])
            })

        if pose is not None:

            try:

                left_elbow_angle = detector.angle(
                    pose["left_shoulder"],
                    pose["left_elbow"],
                    pose["left_wrist"]
                )

                right_elbow_angle = detector.angle(
                    pose["right_shoulder"],
                    pose["right_elbow"],
                    pose["right_wrist"]
                )

                features = detector.build_features(
                    pose,
                    objects_boxes
                )

                frame_data["pose_features"] = {

                    "left_elbow_angle":
                        float(left_elbow_angle),

                    "right_elbow_angle":
                        float(right_elbow_angle),

                    "left_wrist_distance":
                        None if features is None
                        else float(features[0]),

                    "right_wrist_distance":
                        None if features is None
                        else float(features[1]),

                    "left_elbow_distance":
                        None if features is None
                        else float(features[2]),

                    "right_elbow_distance":
                        None if features is None
                        else float(features[3])
                }
                if frame_id % 500 == 0:
                  print(f"Processed frame: {frame_id}")

            except Exception as e:

                print(
                    f"🔴Pose feature error at frame {frame_id}:",
                    e
                )




        sequence_data["frames"].append(
            frame_data
        )

        frame_id += 1

    cap.release()



    all_sequences.append(
        sequence_data
    )


print(
    f"Extracted {len(all_sequences)} sequence(s)."
)

print(
    f"Frames in first sequence: "
    f"{len(all_sequences[0]['frames'])}"
    if all_sequences
    else "No sequences extracted."
)

{'face_count': 2}
{'pitch': -179.85036283261076, 'yaw': 2.216227859007766, 'roll': 6.772974327442393, 'rvec': [-3.135201071265221, -0.1854437818117349, 0.06088526271383784], 'tvec': [505.7527536385029, 319.04139122080596, 2766.4112366334293], 'rotation_matrix': [[0.9922784711005334, 0.11783489388557009, -0.038708830689702496], [0.11784737088744227, -0.9930297715697978, -0.0019672190911864916], [-0.03867082835038888, -0.0026097047747862655, -0.9992485959338062]]}


KeyboardInterrupt: 

In [ ]:
import pickle

with open("/content/drive/MyDrive/Online_Exam_Cheating/.pkl", "wb") as f:
    pickle.dump(all_sequences, f)

print(all_sequences[0])

Output hidden; open in https://colab.research.google.com to view.